# CENG428 Neural Networks — Synthetic Shape Detection

**Framework:** PyTorch + torchvision  
**Base dataset:** MS COCO 2017  
**Task:** Detect or segment synthetic shapes added to natural images

In [ ]:
import hashlib
import random
from pathlib import Path
from typing import Literal

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import CocoDetection
import matplotlib.pyplot as plt

print("PyTorch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

---
## 2. Dataset and Split

In [ ]:
# ── Constants given by the assignment ──────────────────────────────
GLOBAL_SEED          = 2025
TRAIN_SIZE           = 5000   # first 5000 images from train2017
VAL_SIZE             = 1000   # first 1000 images from val2017
TEST_SIZE            = 1000   # next  1000 images from val2017 (indices 1000–1999)
POSITIVE_RATIO       = 0.70   # 70% positive, 30% negative
MAX_SHAPES_PER_IMAGE = 3      # 1 to 3 shapes per positive image


# ── Deterministic seed — given by the assignment ───────────────────
# Do NOT use Python's built-in hash(); output varies between sessions.
def make_seed(split_name: str, image_id: int, global_seed: int = GLOBAL_SEED) -> int:
    key = f"{split_name}_{image_id}_{global_seed}".encode("utf-8")
    return int(hashlib.sha256(key).hexdigest()[:8], 16)


# ── COCO base initialization — given by the assignment ─────────────
# Required directory structure:
#   data/coco/
#   ├── train2017/
#   ├── val2017/
#   └── annotations/
#       ├── instances_train2017.json
#       └── instances_val2017.json
def build_coco_bases(data_root: str | Path = "data/coco"):
    root = Path(data_root)

    train_base = CocoDetection(
        root=str(root / "train2017"),
        annFile=str(root / "annotations" / "instances_train2017.json"),
    )

    val_base = CocoDetection(
        root=str(root / "val2017"),
        annFile=str(root / "annotations" / "instances_val2017.json"),
    )

    return train_base, val_base


# ── Fixed split protocol — given by the assignment ─────────────────
#   - Sort all image IDs in increasing order
#   - train : first 5000 from train2017
#   - val   : first 1000 from val2017
#   - test  : next  1000 from val2017  (do NOT use for model selection)
def get_split_ids(
    coco_base: CocoDetection,
    split: Literal["train", "val", "test"],
) -> list[int]:
    all_ids = sorted(coco_base.coco.getImgIds())

    if split == "train":
        return all_ids[:TRAIN_SIZE]
    elif split == "val":
        return all_ids[:VAL_SIZE]
    elif split == "test":
        return all_ids[VAL_SIZE : VAL_SIZE + TEST_SIZE]
    else:
        raise ValueError(f"Unknown split: {split!r}")

In [ ]:
train_base, val_base = build_coco_bases("data/coco")

train_ids = get_split_ids(train_base, "train")   # first 5000 from train2017
val_ids   = get_split_ids(val_base,   "val")     # first 1000 from val2017
test_ids  = get_split_ids(val_base,   "test")    # next  1000 from val2017

print(f"Train: {len(train_ids)}  Val: {len(val_ids)}  Test: {len(test_ids)}")

---
## 3. Synthetic Shape Generation

Shapes to support (§4): circles, rectangles, triangles, ellipses, polygons, line segments, stars or simple icons.

Properties to vary per shape (§4): type, location, size, color, opacity, rotation, number of shapes per image.

At least **4** difficulty mechanisms required (§4):
random opacity, anti-aliased edges, blur, noise, low-contrast colors, colors from local image stats,
partial transparency, overlapping shapes, random resize/crop, hard negatives, distractor shapes.

In [ ]:
class ShapeGenerator:
    def generate(self, image, n_shapes: int, rng: random.Random):
        """
        Draw n_shapes synthetic shapes onto image.

        Args:
            image    : PIL.Image (RGB)
            n_shapes : number of shapes to draw (0 for negative images)
            rng      : seeded random.Random instance

        Returns:
            augmented_image : PIL.Image
            boxes           : list[list[float]]   [[x1, y1, x2, y2], ...]
            mask            : np.ndarray [H, W]   1=shape pixel, 0=background
        """
        raise NotImplementedError

In [ ]:
# Visualize at least 12 generated examples (§4).
# Must include both positive and negative samples.
raise NotImplementedError

---
## 4. Label Generation

Choose **Option A** (Object Detection) or **Option B** (Semantic Segmentation).

A classification-only model is **not sufficient** for full credit (§5).

In [ ]:
# ── PyTorch Dataset wrapper — must implement this ──────────────────
#
# The wrapper must (§2):
#   1. Load a COCO image
#   2. Add one or more synthetic shapes
#   3. Generate the corresponding target label automatically
#   4. Return the modified image and generated target
#
# Target format — Option A: Object Detection (§5)
#   target = {
#       "boxes":       FloatTensor[N, 4]   # x_min, y_min, x_max, y_max
#       "labels":      LongTensor[N]        # all synthetic shapes share label 1
#       "image_id":    int
#       "is_positive": bool
#   }
#   If negative: boxes = zeros((0,4)), labels = zeros((0,))
#
# Target format — Option B: Semantic Segmentation (§5)
#   target = {
#       "mask":        LongTensor or FloatTensor[H, W]   # 1=shape, 0=background
#       "image_id":    int
#       "is_positive": bool
#   }
#   If negative: mask contains only zeros
class SyntheticShapeDataset(Dataset):
    def __init__(
        self,
        coco_base: CocoDetection,
        image_ids: list[int],
        split: Literal["train", "val", "test"],
        task: Literal["detection", "segmentation"],
        transform=None,
    ):
        self.coco_base = coco_base
        self.image_ids = image_ids
        self.split     = split
        self.task      = task
        self.transform = transform
        self.generator = ShapeGenerator()

    def __len__(self) -> int:
        return len(self.image_ids)

    def __getitem__(self, idx: int):
        image_id = self.image_ids[idx]

        # Seed: random for train, deterministic for val/test (§3)
        if self.split == "train":
            rng = random.Random()
        else:
            rng = random.Random(make_seed(self.split, image_id))

        # Positive / negative decision — 70% positive, 30% negative (§4)
        is_positive = rng.random() < POSITIVE_RATIO
        n_shapes    = rng.randint(1, MAX_SHAPES_PER_IMAGE) if is_positive else 0

        # Step 1: Load a COCO image
        # Step 2: Add synthetic shapes
        # Step 3: Generate target label automatically
        # Step 4: Return (image, target)
        raise NotImplementedError

---
## 5. Model Architecture

Use a CNN-based architecture (§6). May train from scratch or fine-tune a pretrained model.
Clearly state which parts are pretrained and which are trained by you.

Detection options: Faster R-CNN, SSD-style, YOLO-style, custom CNN detector.  
Segmentation options: U-Net, FCN-style, encoder-decoder CNN, custom CNN.

In [ ]:
def build_model(task: str, pretrained: bool):
    """
    Build and return a CNN-based detection or segmentation model.
    Clearly state which parts are pretrained and which are trained from scratch.
    """
    raise NotImplementedError


TASK      = "detection"   # or "segmentation"
PRETRAINED = True
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"

model = build_model(TASK, PRETRAINED).to(DEVICE)

---
## 6. Training Procedure

Required training details to report (§7):

| Detail | Value |
|--------|-------|
| Input image size | |
| Batch size | |
| Number of epochs | |
| Optimizer | |
| Learning rate | |
| Loss function | |
| Pretrained weights | |
| Hardware | |
| Approximate training time | |

In [ ]:
import torchvision.transforms as T

def get_transforms(split: str):
    """Return transforms for the given split. At minimum: Resize, ToTensor, Normalize."""
    raise NotImplementedError


train_ds = SyntheticShapeDataset(train_base, train_ids, "train", TASK, get_transforms("train"))
val_ds   = SyntheticShapeDataset(val_base,   val_ids,   "val",   TASK, get_transforms("val"))

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=4)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


def train_one_epoch(model, loader, optimizer, device) -> float:
    """Run one training epoch; return average loss."""
    raise NotImplementedError


def evaluate_val(model, loader, device) -> dict:
    """Evaluate on validation set; return metric dict. Do NOT use on test set for model selection."""
    raise NotImplementedError


import time, json
history = []

for epoch in range(1, 11):
    t0 = time.time()
    train_loss  = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_metrics = evaluate_val(model, val_loader, DEVICE)
    elapsed     = time.time() - t0

    print(f"Epoch {epoch:02d}  loss={train_loss:.4f}  {val_metrics}  ({elapsed:.1f}s)")
    history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})

Path("results").mkdir(exist_ok=True)
Path("results/metrics.json").write_text(json.dumps(history, indent=2))

---
## 7. Baseline Method

Compare the CNN model against at least one simple baseline (§8).
Options: color thresholding, edge detection + connected components,
template matching, logistic regression on handcrafted features, or shallow CNN.

In [ ]:
class Baseline:
    """
    Simple baseline to compare against the CNN model (§8).
    Choose one of the options listed in §8.
    """

    def predict(self, image):
        raise NotImplementedError

---
## 8. Evaluation Metrics

In [ ]:
# Fixed test split — given by the assignment
# Do NOT use for model selection (§7)
test_ds     = SyntheticShapeDataset(val_base, test_ids, "test", TASK, get_transforms("val"))
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=4)


def detection_metrics(predictions: list, targets: list, iou_threshold: float = 0.5) -> dict:
    """
    Required (§9): Precision@0.5, Recall@0.5, F1@0.5, mean IoU of matched predictions.
    Match predicted boxes to ground-truth boxes via greedy IoU matching.
    """
    raise NotImplementedError


def segmentation_metrics(predictions: list, targets: list) -> dict:
    """
    Required (§9): foreground IoU, Dice coefficient, foreground precision, foreground recall.
    Pixel accuracy alone is NOT sufficient.
    """
    raise NotImplementedError


model.eval()
all_predictions, all_targets, all_images = [], [], []

with torch.no_grad():
    for images, targets in test_loader:
        pass  # collect predictions

if TASK == "detection":
    test_metrics = detection_metrics(all_predictions, all_targets)
else:
    test_metrics = segmentation_metrics(all_predictions, all_targets)

print("Test metrics:", test_metrics)

---
## 9. Experiments and Results

At least **3** meaningful experiments required (§10).  
For each: report quantitative results **and** a short interpretation (do not only list numbers).

In [ ]:
# Experiment 1 — e.g. training from scratch vs fine-tuning (§10.1)
raise NotImplementedError

In [ ]:
# Experiment 2 — e.g. small vs larger training set (§10.2)
raise NotImplementedError

In [ ]:
# Experiment 3 — e.g. easy vs hard synthetic shapes (§10.3)
raise NotImplementedError

---
## 10. Prediction Visualizations

At least **12** test images required (§9).  
Must include: successful predictions, failure cases, positive images, negative images.

In [ ]:
def visualize_predictions(images, targets, predictions, n: int = 12, save_dir: str = "figures"):
    """
    Visualize at least 12 test images (§9).
    Include: successful predictions, failure cases, positive images, negative images.
    """
    raise NotImplementedError

visualize_predictions(all_images, all_targets, all_predictions, n=12)

---
## 11. Failure Cases

Briefly discuss which types of synthetic additions are difficult for the model to detect and why.

---
## 12. Conclusion

Summarize findings, limitations, and possible improvements.